# Cassava vs Plantain vs Other — Image Classifier

Trains a YOLOv8 classification model on your manually-sorted dataset.

**Expected folder structure (zipped and uploaded, or on Google Drive):**
```
dataset/
├── train/
│   ├── cassava/
│   ├── plantain/
│   └── other/
└── val/
    ├── cassava/
    ├── plantain/
    └── other/
```

If you only have ONE folder per class (no train/val split yet), run the optional split cell below first.

## 1. Set up GPU runtime
Runtime -> Change runtime type -> select **T4 GPU** (or better) before running anything below.

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q ultralytics

## 3. Get your dataset into Colab

**Option A — Upload a zip directly (simplest for small datasets)**

Zip your `dataset` folder locally first (must contain `train/` and `val/` subfolders, each with `cassava/`, `plantain/`, `other/`), then run the cell below and select the zip when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select your dataset.zip

import zipfile, os
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

print('Extracted to /content/dataset')
!find /content/dataset -maxdepth 3 -type d

**Option B — Mount Google Drive instead (better for larger datasets / repeated runs)**

Skip this if you used Option A above. Uncomment and edit the path to match where you uploaded `dataset/` in your Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = '/content/drive/MyDrive/dataset'  # edit to match your Drive path
# print('Using dataset at:', DATASET_PATH)

## 3b. (Optional) Auto-split into train/val

Only run this if your images are currently in flat class folders (`cassava/`, `plantain/`, `other/`) with **no** train/val split yet. Skip if you already split manually.

In [ ]:
import shutil, random
from pathlib import Path

SOURCE_DIR = Path('/content/dataset_flat')   # folder containing cassava/, plantain/, other/ with no split
OUT_DIR = Path('/content/dataset')           # where the train/val split will be written
VAL_SPLIT = 0.2
SEED = 42

random.seed(SEED)
classes = [d.name for d in SOURCE_DIR.iterdir() if d.is_dir()]
print('Classes found:', classes)

for cls in classes:
    files_list = list((SOURCE_DIR / cls).glob('*'))
    random.shuffle(files_list)
    n_val = max(1, int(len(files_list) * VAL_SPLIT))
    val_files = files_list[:n_val]
    train_files = files_list[n_val:]

    for split_name, split_files in [('train', train_files), ('val', val_files)]:
        dest = OUT_DIR / split_name / cls
        dest.mkdir(parents=True, exist_ok=True)
        for f in split_files:
            shutil.copy(f, dest / f.name)

    print(f'{cls}: {len(train_files)} train, {len(val_files)} val')

## 4. Sanity check the dataset before training

In [ ]:
from pathlib import Path

DATASET_PATH = '/content/dataset'  # update if using Drive path from Option B

for split in ['train', 'val']:
    split_dir = Path(DATASET_PATH) / split
    print(f'--- {split} ---')
    for cls_dir in sorted(split_dir.iterdir()):
        if cls_dir.is_dir():
            count = len(list(cls_dir.glob('*')))
            print(f'  {cls_dir.name}: {count} images')

## 5. Train

`yolov8n-cls.pt` is the smallest/fastest pretrained classification checkpoint — good starting point given your dataset size. Swap to `yolov8s-cls.pt` or `yolov8m-cls.pt` later if you have more data and want higher accuracy at the cost of speed.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')

results = model.train(
    data=DATASET_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,       # stop early if val accuracy stalls
    project='cassava_plantain_run',
    name='exp1'
)

## 6. Validate

In [ ]:
metrics = model.val()
print(metrics)

## 7. Test on a single image

In [ ]:
from google.colab import files
test_upload = files.upload()  # upload a test image
test_path = list(test_upload.keys())[0]

result = model(test_path)
result[0].show()
print('Predicted class:', result[0].names[result[0].probs.top1])
print('Confidence:', result[0].probs.top1conf.item())

## 8. Download the trained weights

Best checkpoint is saved at `cassava_plantain_run/exp1/weights/best.pt`.

In [ ]:
from google.colab import files
files.download('cassava_plantain_run/exp1/weights/best.pt')